In [1]:
!pip install "torchao>=0.16.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 56.7 MB/s eta 0:00:00:00:01
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


In [2]:
# Initialize Git Large File Storage
!git lfs install

# Clone the repository directly to your Kaggle working directory
!git clone https://huggingface.co/hugohrban/progen2-small /kaggle/working/progen2-small-local

print("Download complete! The model is physically on your Kaggle disk.")

Git LFS initialized.
Cloning into '/kaggle/working/progen2-small-local'...
remote: Enumerating objects: 68, done.
remote: Total 68 (delta 0), reused 0 (delta 0), pack-reused 68 (from 1)
Receiving objects: 100% (68/68), 37.26 KiB | 6.21 MiB/s, done.
Resolving deltas: 100% (32/32), done.
Download complete! The model is physically on your Kaggle disk.


In [ ]:
script=r'''#!/usr/bin/env python
"""
LoRA SFT of ProGen2 with DDP support + Kaggle 8h chaining.
Scientifically aligned with DPO/KTO baselines.
"""

# --------------------------------------------------------------------------- #
# 0. Environment guards
# --------------------------------------------------------------------------- #
import os
os.environ["TORCHDYNAMO_DISABLE"] = "1"
os.environ["TORCH_COMPILE_DISABLE"] = "1"

# --------------------------------------------------------------------------- #
# 1. Compatibility Patches (PEFT / TorchAO / TensorParallel)
# --------------------------------------------------------------------------- #
try:
    import peft.import_utils
    # Overwrite the root check
    peft.import_utils.is_torchao_available = lambda: False
    
    # Forcefully overwrite the bound reference deep inside the LoRA module
    import peft.tuners.lora.torchao
    peft.tuners.lora.torchao.is_torchao_available = lambda: False
except Exception:
    pass

try:
    import transformers.integrations.tensor_parallel as tp
    if not hasattr(tp, "EmbeddingParallel"):
        tp.EmbeddingParallel = type("EmbeddingParallel", (object,), {})
except ImportError:
    pass

# --------------------------------------------------------------------------- #
# 2. Imports
# --------------------------------------------------------------------------- #
import csv
import logging
import sys
import time

import pandas as pd
import torch
from datasets import Dataset
import datasets
datasets.utils.logging.disable_progress_bar()

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    DataCollatorForLanguageModeling,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
    set_seed,
)
from transformers.trainer_callback import TrainerCallback, ProgressCallback, PrinterCallback
from transformers.trainer_utils import get_last_checkpoint
from peft import LoraConfig, TaskType, get_peft_model

logging.basicConfig(
    format="%(asctime)s | %(levelname)-8s | %(name)s | %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
    level=logging.INFO,
    handlers=[logging.StreamHandler(sys.stdout)],
)
logger = logging.getLogger("lora_sft")

# --------------------------------------------------------------------------- #
# Callbacks
# --------------------------------------------------------------------------- #
class TimeLimitCallback(TrainerCallback):
    def __init__(self, time_limit_hours=10.0):
        self.time_limit_sec = time_limit_hours * 3600
        self.start_time = None

    def on_train_begin(self, args, state, control, **kwargs):
        if self.start_time is None:
            self.start_time = time.time()

    def on_step_end(self, args, state, control, **kwargs):
        if self.start_time and (time.time() - self.start_time > self.time_limit_sec):
            if state.is_world_process_zero:
                logger.info(f"\n[TimeLimitCallback] Reached {self.time_limit_sec/3600} hours. Stopping gracefully.")
            control.should_training_stop = True

def _render_bar(pct, width=30):
    filled = int(width * pct / 100)
    return "[" + ("#" * filled) + ("-" * (width - filled)) + f"] {pct:3d}%"

def _fmt_time(seconds):
    seconds = max(0, int(seconds))
    h, rem = divmod(seconds, 3600)
    m, s = divmod(rem, 60)
    return f"{h:02d}:{m:02d}:{s:02d}" if h else f"{m:02d}:{s:02d}"

def _eta(elapsed, done, total):
    if done <= 0 or total <= 0 or done >= total:
        return "--:--:--"
    rate = elapsed / done
    return _fmt_time(rate * (total - done))

class SingleLineProgressCallback(TrainerCallback):
    MIN_INTERVAL = 2.0  
    def __init__(self):
        self.eval_total_steps = None
        self.train_start_time = None
        self.train_last_print_time = 0.0
        self.last_train_logs = {}
        self.eval_count = 0
        self.eval_start_time = None
        self.eval_last_print_time = 0.0

    def _print(self, line):
        sys.stdout.write("\r" + line.ljust(120))
        sys.stdout.flush()

    def _render_train_line(self, state, pct):
        elapsed = time.time() - self.train_start_time if self.train_start_time else 0
        epoch = state.epoch or 0.0
        max_steps = int(state.max_steps) if state.max_steps else 0
        eta = _eta(elapsed, state.global_step, max_steps)
        extras = ""
        
        loss = self.last_train_logs.get("loss")
        lr = self.last_train_logs.get("learning_rate")
        if loss is not None:
            extras += f"  loss {loss:.4f}"
        if lr is not None:
            extras += f"  lr {lr:.2e}"
            
        return (
            f"Training   {_render_bar(pct)}  "
            f"step {state.global_step}/{max_steps}  "
            f"epoch {epoch:.2f}  "
            f"elapsed {_fmt_time(elapsed)}  eta {eta}{extras}"
        )

    def on_train_begin(self, args, state, control, **kwargs):
        if state.is_world_process_zero:
            self.train_start_time = time.time()
            self.train_last_print_time = time.time()
            self._print(self._render_train_line(state, 0))

    def on_step_end(self, args, state, control, **kwargs):
        if not state.is_world_process_zero or not state.max_steps:
            return
        now = time.time()
        is_last_step = state.global_step >= state.max_steps
        if is_last_step or (now - self.train_last_print_time) >= self.MIN_INTERVAL:
            self.train_last_print_time = now
            pct = min(int(state.global_step / state.max_steps * 100), 100)
            self._print(self._render_train_line(state, pct))

    def on_train_end(self, args, state, control, **kwargs):
        if state.is_world_process_zero:
            self._print(self._render_train_line(state, 100))
            sys.stdout.write("\n")
            sys.stdout.flush()

    def on_log(self, args, state, control, logs=None, **kwargs):
        if state.is_world_process_zero and logs and "loss" in logs:
            sys.stdout.write("\n")
            sys.stdout.flush()
            clean_logs = {k: f"{v:.6f}" if isinstance(v, float) else v for k, v in logs.items() if not k.startswith("epoch") and not k.startswith("step")}
            print(f"Metrics: {clean_logs}")
            self.last_train_logs = logs
            
            max_steps = int(state.max_steps) if state.max_steps else 0
            pct = min(int(state.global_step / max_steps * 100), 100) if max_steps else 0
            self.train_last_print_time = time.time()
            self._print(self._render_train_line(state, pct))

    def on_prediction_step(self, args, state, control, **kwargs):
        if not state.is_world_process_zero:
            return
        if self.eval_count == 0:
            sys.stdout.write("\n")
            sys.stdout.flush()
            self.eval_start_time = time.time()
            self.eval_last_print_time = 0.0
        self.eval_count += 1
        now = time.time()
        elapsed = now - self.eval_start_time
        is_last_batch = self.eval_total_steps and self.eval_count >= self.eval_total_steps
        if is_last_batch or (now - self.eval_last_print_time) >= self.MIN_INTERVAL:
            self.eval_last_print_time = now
            if self.eval_total_steps:
                pct = min(int(self.eval_count / self.eval_total_steps * 100), 100)
                eta = _eta(elapsed, self.eval_count, self.eval_total_steps)
                self._print(
                    f"Evaluating {_render_bar(pct)}  "
                    f"batch {self.eval_count}/{self.eval_total_steps}  "
                    f"elapsed {_fmt_time(elapsed)}  eta {eta}"
                )
            else:
                self._print(f"Evaluating...  batch {self.eval_count}  elapsed {_fmt_time(elapsed)}")

    def on_evaluate(self, args, state, control, metrics=None, **kwargs):
        if state.is_world_process_zero:
            if metrics and self.eval_count > 0:
                sys.stdout.write("\n")
                sys.stdout.flush()
                clean_metrics = {k: f"{v:.6f}" if isinstance(v, float) else v for k, v in metrics.items()}
                print(f"Eval Metrics: {clean_metrics}")
            elapsed = time.time() - self.eval_start_time if self.eval_start_time else 0
            total = self.eval_total_steps or self.eval_count
            extra = ""
            if metrics and metrics.get("eval_loss") is not None:
                extra = f"  eval_loss {metrics['eval_loss']:.6f}"
            self._print(
                f"Evaluating {_render_bar(100)}  "
                f"batch {self.eval_count}/{total}  "
                f"elapsed {_fmt_time(elapsed)}{extra}"
            )
            sys.stdout.write("\n")
            sys.stdout.flush()
            self.eval_count = 0
            self.train_last_print_time = 0.0

class MetricsFileLoggerCallback(TrainerCallback):
    def __init__(self, output_dir):
        self.log_path = os.path.join(output_dir, "metrics_log.csv")
        self._fieldnames = None

    def on_log(self, args, state, control, logs=None, **kwargs):
        if not state.is_world_process_zero or not logs:
            return
        row = {"step": state.global_step, "epoch": round(state.epoch, 4) if state.epoch is not None else None}
        row.update(logs)
        
        file_exists = os.path.exists(self.log_path)
        if self._fieldnames is None:
            if file_exists:
                with open(self.log_path, "r", newline="") as f:
                    existing_header = next(csv.reader(f), [])
                self._fieldnames = existing_header if existing_header else list(row.keys())
            else:
                self._fieldnames = list(row.keys())

        new_keys = [k for k in row.keys() if k not in self._fieldnames]
        if new_keys:
            self._fieldnames = self._fieldnames + new_keys
            if file_exists:
                with open(self.log_path, "r", newline="") as f:
                    existing_rows = list(csv.DictReader(f))
                with open(self.log_path, "w", newline="") as f:
                    writer = csv.DictWriter(f, fieldnames=self._fieldnames)
                    writer.writeheader()
                    for r in existing_rows:
                        writer.writerow(r)
        
        with open(self.log_path, "a", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=self._fieldnames)
            if not file_exists and not new_keys:
                writer.writeheader()
            writer.writerow(row)

# --------------------------------------------------------------------------- #
# 3. Parameters
# --------------------------------------------------------------------------- #
TRAIN_CSV = "/kaggle/input/datasets/nasiatfahim/thesis-dataset-1/train.csv"
TEST_CSV = "/kaggle/input/datasets/nasiatfahim/thesis-dataset-1/test.csv"
SEQ_COL = "Sequence"
MODEL_NAME = "/kaggle/working/progen2-small-local"
OUTPUT_DIR = "/kaggle/working/out-lora-sft"
MAX_LENGTH = 512
EPOCHS = 5
LR = 1e-4
BATCH_SIZE = 8
GRAD_ACCUM = 4
LORA_R = 8
LORA_ALPHA = 16
LORA_DROPOUT = 0.05
LORA_TARGET_MODULES = "all-linear"
EARLY_STOPPING_PATIENCE = 10
EVAL_STEPS = 100
SAVE_TOTAL_LIMIT = 3
TIME_LIMIT_HOURS = 6.0  # <--- UPDATED to 8 hours
SEED = 42

def _is_bf16_available():
    if not torch.cuda.is_available(): return False
    try: return torch.cuda.is_bf16_supported()
    except Exception: return False
BF16 = _is_bf16_available()

def _is_ddp_initialized():
    if torch.distributed.is_initialized(): return True
    return "RANK" in os.environ and "WORLD_SIZE" in os.environ

class Args:
    def __init__(self):
        self.train_csv = TRAIN_CSV
        self.test_csv = TEST_CSV
        self.seq_col = SEQ_COL
        self.model_name = MODEL_NAME
        self.output_dir = OUTPUT_DIR
        self.max_length = MAX_LENGTH
        self.epochs = EPOCHS
        self.lr = LR
        self.batch_size = BATCH_SIZE
        self.grad_accum = GRAD_ACCUM
        self.lora_r = LORA_R
        self.lora_alpha = LORA_ALPHA
        self.lora_dropout = LORA_DROPOUT
        self.lora_target_modules = LORA_TARGET_MODULES
        self.early_stopping_patience = EARLY_STOPPING_PATIENCE
        self.eval_steps = EVAL_STEPS
        self.save_total_limit = SAVE_TOTAL_LIMIT
        self.bf16 = BF16
        self.gradient_checkpointing = False
        self.seed = SEED
        self.time_limit_hours = TIME_LIMIT_HOURS

def parse_args():
    return Args()

# --------------------------------------------------------------------------- #
# 4. Data helpers
# --------------------------------------------------------------------------- #
def load_data(train_csv, test_csv, seq_col):
    logger.info(f"Loading '{train_csv}' ...")
    train_df = pd.read_csv(train_csv)
    if seq_col not in train_df.columns:
        raise ValueError(f"Column '{seq_col}' not found in {train_csv}")

    train_df = train_df[[seq_col]].dropna().drop_duplicates().reset_index(drop=True)
    logger.info(f"Loaded {len(train_df)} unique train sequences")

    logger.info(f"Loading '{test_csv}' ... ONLY using 1500")
    test_df = pd.read_csv(test_csv)
    if seq_col not in test_df.columns:
        raise ValueError(f"Column '{seq_col}' not found in {test_csv}")

    # Deduplicate BEFORE slicing
    test_df = test_df[[seq_col]].dropna().drop_duplicates().reset_index(drop=True)
    test_df = test_df[:1500] 
    logger.info(f"Loaded {len(test_df)} unique test sequences")

    return Dataset.from_pandas(train_df), Dataset.from_pandas(test_df)

def build_tokenizer(model_name):
    tok = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
    if tok.pad_token is None:
        tok.add_special_tokens({"pad_token": "<|pad|>"})
        logger.info("Added distinct <|pad|> token to tokenizer.")
    assert tok.eos_token_id is not None, "Tokenizer has no EOS token. Training cannot proceed."
    return tok

# --------------------------------------------------------------------------- #
# EMPIRICAL PROBES (Double Verification)
# --------------------------------------------------------------------------- #
def verify_masks(model):
    model.eval()
    vocab_size = model.get_input_embeddings().weight.shape[0]
    
    # --- PROBE 1: CAUSALITY (Future Leakage) ---
    seq_len = 24
    input_ids_causal_a = torch.randint(low=4, high=min(vocab_size, 50), size=(1, seq_len)).to(model.device)
    input_ids_causal_b = input_ids_causal_a.clone()
    
    split_point = seq_len // 2
    input_ids_causal_b[0, split_point:] = (input_ids_causal_b[0, split_point:] + 7) % min(vocab_size, 50)
    input_ids_causal_b[0, split_point:] = input_ids_causal_b[0, split_point:].clamp(min=4)

    with torch.no_grad():
        logits_causal_a = model(input_ids=input_ids_causal_a).logits
        logits_causal_b = model(input_ids=input_ids_causal_b).logits

    past_logits_a = logits_causal_a[0, :split_point, :]
    past_logits_b = logits_causal_b[0, :split_point, :]
    max_diff_causal = (past_logits_a - past_logits_b).abs().max().item()
    
    if max_diff_causal > 1e-3:
        raise RuntimeError(f"CAUSALITY LEAK DETECTED! max |Δlogits| = {max_diff_causal:.6f}")

    # --- PROBE 2: PADDING (Attention Mask Leakage) ---
    real_len = 16
    pad_len = 8
    input_ids_pad_a = torch.randint(low=4, high=min(vocab_size, 50), size=(1, real_len + pad_len)).to(model.device)
    input_ids_pad_b = input_ids_pad_a.clone()
    
    input_ids_pad_b[0, real_len:] = (input_ids_pad_b[0, real_len:] + 7) % min(vocab_size, 50)
    input_ids_pad_b[0, real_len:] = input_ids_pad_b[0, real_len:].clamp(min=4)
    
    attn_mask = torch.ones((1, real_len + pad_len), dtype=torch.long).to(model.device)
    attn_mask[0, real_len:] = 0

    with torch.no_grad():
        logits_pad_a = model(input_ids=input_ids_pad_a, attention_mask=attn_mask).logits
        logits_pad_b = model(input_ids=input_ids_pad_b, attention_mask=attn_mask).logits

    real_logits_a = logits_pad_a[0, :real_len, :]
    real_logits_b = logits_pad_b[0, :real_len, :]
    max_diff_pad = (real_logits_a - real_logits_b).abs().max().item()

    if max_diff_pad > 1e-3:
        raise RuntimeError(f"PADDING LEAK DETECTED! Attention mask is ignored. max |Δlogits| = {max_diff_pad:.6f}")

    model.train()
    return max_diff_causal, max_diff_pad

# --------------------------------------------------------------------------- #
# 6. Main training loop
# --------------------------------------------------------------------------- #
def main():
    args = parse_args()
    set_seed(args.seed)
    os.makedirs(args.output_dir, exist_ok=True)
    logger.info(f"Args: {vars(args)}")

    local_rank = int(os.environ.get("LOCAL_RANK", 0))
    if torch.cuda.is_available():
        torch.cuda.set_device(local_rank)
        logger.info(f"Process {local_rank}: set CUDA device to {local_rank}")

    if torch.cuda.device_count() > 1 and not _is_ddp_initialized():
        logger.warning("GPUs detected but DDP is NOT initialized. Fallback to DataParallel.")

    tokenizer = build_tokenizer(args.model_name)
    train_ds, val_ds = load_data(args.train_csv, args.test_csv, args.seq_col)

    def tokenize_fn(batch):
        tokenized = tokenizer(
            batch[args.seq_col],
            truncation=True,
            max_length=args.max_length - 1,
            padding=False,
        )
        for i in range(len(tokenized["input_ids"])):
            tokenized["input_ids"][i].append(tokenizer.eos_token_id)
            if "attention_mask" in tokenized:
                tokenized["attention_mask"][i].append(1)
        return tokenized

    logger.info("Tokenizing datasets ...")
    # For SFT we don't strictly need to silence this as it's sequential, but it keeps things clean.
    train_ds = train_ds.map(tokenize_fn, batched=True, remove_columns=train_ds.column_names)
    val_ds = val_ds.map(tokenize_fn, batched=True, remove_columns=val_ds.column_names)

    logger.info(f"Loading base model '{args.model_name}' ...")

    import transformers.modeling_utils
    _orig_mark_tied = transformers.modeling_utils.PreTrainedModel.mark_tied_weights_as_initialized
    def _safe_mark_tied(self, *args, **kwargs):
        try:
            _orig_mark_tied(self, *args, **kwargs)
        except AttributeError as e:
            if "all_tied_weights_keys" in str(e) or "_tied_weights_keys" in str(e):
                pass
            else:
                raise
    transformers.modeling_utils.PreTrainedModel.mark_tied_weights_as_initialized = _safe_mark_tied

    model = AutoModelForCausalLM.from_pretrained(
        args.model_name,
        trust_remote_code=True,
        torch_dtype=torch.bfloat16 if args.bf16 else torch.float32,
        low_cpu_mem_usage=False,
    )

    progen_causal_cls = model.__class__
    progen_base_cls = model.transformer.__class__

    def _get_input_embeddings_causal(self): return self.transformer.wte
    def _set_input_embeddings_causal(self, value): self.transformer.wte = value
    def _get_input_embeddings_base(self): return self.wte
    def _set_input_embeddings_base(self, value): self.wte = value
    def _get_output_embeddings_causal(self): return self.lm_head
    def _set_output_embeddings_causal(self, value): self.lm_head = value

    progen_causal_cls.get_input_embeddings = _get_input_embeddings_causal
    progen_causal_cls.set_input_embeddings = _set_input_embeddings_causal
    if hasattr(model, "lm_head"):
        progen_causal_cls.get_output_embeddings = _get_output_embeddings_causal
        progen_causal_cls.set_output_embeddings = _set_output_embeddings_causal

    progen_base_cls.get_input_embeddings = _get_input_embeddings_base
    progen_base_cls.set_input_embeddings = _set_input_embeddings_base

    if len(tokenizer) > model.get_input_embeddings().weight.shape[0]:
        model.resize_token_embeddings(len(tokenizer))
        model.tie_weights()

    def _fix_progen2_scale_attn_safe(module):
        for name, child in module.named_modules():
            if not hasattr(child, "scale_attn"): continue
            if "scale_attn" in child._buffers: continue
            attr = child.scale_attn
            if not isinstance(attr, torch.Tensor): continue
            if attr.device.type == "meta":
                state = child.state_dict()
                if "scale_attn" in state:
                    new_scale = state["scale_attn"]
                else:
                    head_dim = getattr(child, "head_dim", None)
                    if head_dim is None:
                        head_dim = getattr(child, "embed_dim", 512) // getattr(child, "num_heads", 8)
                    new_scale = torch.sqrt(torch.tensor(head_dim, dtype=torch.float32))
            else:
                new_scale = attr
            delattr(child, "scale_attn")
            child.register_buffer("scale_attn", new_scale.to(model.dtype))

    _fix_progen2_scale_attn_safe(model)

    def _get_head_mask(self, head_mask, num_hidden_layers, is_attention_chunked=False):
        if head_mask is not None:
            head_mask = self._convert_head_mask_to_5d(head_mask, num_hidden_layers)
            if is_attention_chunked: head_mask = head_mask.unsqueeze(-1)
        else:
            head_mask = [None] * num_hidden_layers
        return head_mask

    def _convert_head_mask_to_5d(self, head_mask, num_hidden_layers):
        if head_mask.dim() == 1:
            head_mask = head_mask.unsqueeze(0).unsqueeze(0).unsqueeze(-1).unsqueeze(-1)
            head_mask = head_mask.expand(num_hidden_layers, -1, -1, -1, -1)
        elif head_mask.dim() == 2:
            head_mask = head_mask.unsqueeze(1).unsqueeze(-1).unsqueeze(-1)
        return head_mask.to(dtype=self.dtype)

    if not hasattr(progen_base_cls, "get_head_mask"):
        progen_base_cls.get_head_mask = _get_head_mask
        progen_base_cls._convert_head_mask_to_5d = _convert_head_mask_to_5d

    if not hasattr(progen_causal_cls, "get_head_mask"):
        progen_causal_cls.get_head_mask = _get_head_mask
        progen_causal_cls._convert_head_mask_to_5d = _convert_head_mask_to_5d

    # ========================================================================= #
    # FIX #19: DEFINITIVE CAUSAL MASK OVERWRITE
    # ========================================================================= #
    model.config.is_decoder = True
    patched_mask_count = 0
    for name, module in model.named_modules():
        buf_names = dict(module.named_buffers(recurse=False))
        if "bias" in buf_names and "masked_bias" in buf_names:
            bias = buf_names["bias"]
            if bias.dim() == 4:
                seq_len = bias.shape[-1]
                causal_mask = torch.tril(torch.ones((seq_len, seq_len), dtype=torch.bool)).view(1, 1, seq_len, seq_len)
                bias.copy_(causal_mask)
                
                masked_bias = buf_names["masked_bias"]
                masked_bias.copy_(torch.tensor(-1e4, dtype=masked_bias.dtype))
                patched_mask_count += 1
                
    if patched_mask_count > 0:
        logger.info(f"Overwrote Causal Mask to STRICT lower-triangular on {patched_mask_count} attention blocks.")

    # ========================================================================= #
    # DOUBLE EMPIRICAL SANITY CHECK
    # ========================================================================= #
    if local_rank == 0:
        c_diff, p_diff = verify_masks(model)
        logger.info(f"✅ MASK VERIFICATION PASSED. Causal leak: {c_diff:.2e} | Pad leak: {p_diff:.2e}")

    target_modules = (
        "all-linear"
        if args.lora_target_modules == "all-linear"
        else [m.strip() for m in args.lora_target_modules.split(",")]
    )

    lora_config = LoraConfig(
        r=args.lora_r,
        lora_alpha=args.lora_alpha,
        lora_dropout=args.lora_dropout,
        bias="none",
        task_type=TaskType.CAUSAL_LM,
        target_modules=target_modules,
    )

    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()

    collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

    training_args = TrainingArguments(
        output_dir=args.output_dir,
        num_train_epochs=args.epochs,
        per_device_train_batch_size=args.batch_size,
        per_device_eval_batch_size=args.batch_size,
        gradient_accumulation_steps=args.grad_accum,
        learning_rate=args.lr,
        warmup_ratio=0.03,
        lr_scheduler_type="cosine",
        logging_dir=os.path.join(args.output_dir, "logs"),
        logging_steps=20,
        eval_strategy="steps",
        eval_steps=args.eval_steps,
        save_strategy="steps",
        save_steps=args.eval_steps,
        save_total_limit=args.save_total_limit,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        bf16=args.bf16,
        report_to=["none"],
        disable_tqdm=True,
        seed=args.seed,
        ddp_find_unused_parameters=False,
        ddp_broadcast_buffers=False,
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        data_collator=collator,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=args.early_stopping_patience)],
    )

    trainer.remove_callback(ProgressCallback)
    trainer.remove_callback(PrinterCallback)

    trainer.add_callback(TimeLimitCallback(time_limit_hours=args.time_limit_hours))
    trainer.add_callback(SingleLineProgressCallback())
    trainer.add_callback(MetricsFileLoggerCallback(args.output_dir))

    if trainer.is_world_process_zero():
        logger.info(f"All train/eval scores will be logged to: {os.path.join(args.output_dir, 'metrics_log.csv')}")

    last_checkpoint = None
    if os.path.isdir(args.output_dir):
        last_checkpoint = get_last_checkpoint(args.output_dir)

    if last_checkpoint is not None and trainer.is_world_process_zero():
        logger.info(f"🔄 Checkpoint detected! Resuming training from: {last_checkpoint}")
    elif trainer.is_world_process_zero():
        logger.info("▶️ Starting new LoRA SFT training from scratch...")

    train_result = trainer.train(resume_from_checkpoint=last_checkpoint)
    logger.info(f"Training complete. Train metrics: {train_result.metrics}")

    logger.info(f"Saving best LoRA adapter to '{args.output_dir}'")
    trainer.save_model(args.output_dir)

    if trainer.is_world_process_zero():
        logger.info(f"Saving tokenizer to '{args.output_dir}'")
        tokenizer.save_pretrained(args.output_dir)

    eval_metrics = trainer.evaluate()
    logger.info(f"Final eval metrics: {eval_metrics}")

if __name__ == "__main__":
    main()
'''

with open("/kaggle/working/train.py", "w") as f:
    f.write(script)

print("Saved to /kaggle/working/train.py")
!torchrun --nproc_per_node=2 /kaggle/working/train.py

Saved to /kaggle/working/train.py
W0714 14:29:24.202000 161 torch/distributed/run.py:852] 
W0714 14:29:24.202000 161 torch/distributed/run.py:852] *****************************************
W0714 14:29:24.202000 161 torch/distributed/run.py:852] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W0714 14:29:24.202000 161 torch/distributed/run.py:852] *****************************************
Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).
Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).
2026-07-14 14:30:07 | INFO     | lora_sft | Args: {'train_csv': '/kaggle/input/datasets/nasiatfahim/thesis-dataset-1/only_therm_non_overlapping_data.csv', 'test_csv': '/kaggle/input/datasets/nasiat